In [0]:
# ============================================================
# NOTEBOOK 03_apply_model
# Vermont EWS — Aplicación del modelo (NO entrena)
#
# Lee modelos persistidos desde Silver/models/
# Aplica clasificador y regresor sobre datos 25-26
# Recalcula clustering dinámico sobre datos actuales
#
# DIFERENCIA vs 03_predictive_model_v2:
# → NO hace Grid Search
# → NO entrena Random Forest
# → NO persiste modelos
# → SÍ recalcula K-Means dinámicamente
# → SÍ aplica clasificador y regresor guardados
# ============================================================

BRONZE  = "/Volumes/workspace/vermont/bronze"
TRUSTED = "/Volumes/workspace/vermont/trusted"
SILVER  = "/Volumes/workspace/vermont/silver"
PRIVADO = "/Volumes/workspace/vermont/privado"

GROUPS = [
    'Science', 'I_and_S', 'Mathematics', 'English',
    'Lengua_Castellana', 'Mandarin', 'Financial_Maths',
    'ICT_STEM', 'Physical_Education', 'Research_Methodology'
]

UMBRAL_PRODUCCION = 0.30
VOLUME_PATH       = f"{SILVER}/models"

import pandas as pd
import numpy as np
import joblib
import json
from datetime import datetime
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

print("=" * 60)
print("APLICACIÓN DE MODELOS — Vermont EWS")
print(f"Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("NO entrena — usa modelos persistidos")
print("=" * 60)

# ── Cargar modelos persistidos ──
print("\n── Cargando modelos desde Silver ──")

rf_clf  = joblib.load(f"{VOLUME_PATH}/rf_classifier.joblib")
imp_clf = joblib.load(f"{VOLUME_PATH}/imputer.joblib")
le      = joblib.load(f"{VOLUME_PATH}/label_encoder.joblib")
rf_reg  = joblib.load(f"{VOLUME_PATH}/rf_regressor.joblib")
imp_reg = joblib.load(f"{VOLUME_PATH}/imputer_reg.joblib")

with open(f"{VOLUME_PATH}/feature_list.json") as f:
    feats_clf = json.load(f)
with open(f"{VOLUME_PATH}/feature_list_reg.json") as f:
    feats_reg = json.load(f)
with open(f"{VOLUME_PATH}/target_list_reg.json") as f:
    T3_COLS = json.load(f)
with open(f"{VOLUME_PATH}/reg_confiabilidad.json") as f:
    confiabilidad = json.load(f)

classes      = le.classes_
idx_critical = list(classes).index('critical')

print(f"✓ Clasificador cargado: {len(feats_clf)} features")
print(f"✓ Regresor cargado:     {len(feats_reg)} features")
print(f"✓ Targets T3:           {len(T3_COLS)} materias")
print(f"✓ Umbral producción:    {UMBRAL_PRODUCCION}")

# ── Cargar datos actuales ──
print("\n── Cargando datos actuales ──")

PREDICT_FE_PATH = f"{TRUSTED}/predict_dataset_fe"
df_predict = spark.read.parquet(PREDICT_FE_PATH).toPandas()
print(f"✓ Estudiantes: {len(df_predict)}")

In [0]:
# ── PASO 1: Clasificación de riesgo ──
print("\n" + "=" * 60)
print("PASO 1 — Clasificación de riesgo")
print("=" * 60)

X_clf = pd.DataFrame(
    imp_clf.transform(df_predict[feats_clf].fillna(0)),
    columns=feats_clf
)

y_proba        = rf_clf.predict_proba(X_clf)
proba_critical = y_proba[:, idx_critical]

# Probabilidad total de riesgo (critical + recovery)
idx_recovery  = list(classes).index('recovery')
proba_recovery = y_proba[:, idx_recovery]
proba_riesgo   = proba_critical + proba_recovery

# Aplicar umbral 0.30
y_pred = []
for i in range(len(y_proba)):
    if y_proba[i, idx_critical] >= UMBRAL_PRODUCCION:
        y_pred.append(idx_critical)
    else:
        proba_resto = y_proba[i].copy()
        proba_resto[idx_critical] = -1
        y_pred.append(np.argmax(proba_resto))

y_pred      = np.array(y_pred)
pred_labels = le.inverse_transform(y_pred)
confianza   = np.max(y_proba, axis=1)

print(f"✓ Predicciones generadas: {len(pred_labels)}")
for cls in classes:
    n = (pred_labels == cls).sum()
    print(f"  {cls}: {n} ({n/len(pred_labels)*100:.1f}%)")

# Calcular T3 parcial acumulada
materias_en_bajo = []
for i in range(len(df_predict)):
    n_bajo = 0
    for g in GROUPS:
        t1 = f'{g}_T1'
        t2 = f'{g}_T2'
        t3 = f'{g}_T3'
        if all(c in df_predict.columns for c in [t1,t2,t3]):
            t1v = df_predict.iloc[i][t1]
            t2v = df_predict.iloc[i][t2]
            t3v = df_predict.iloc[i][t3]
            if not any(pd.isna([t1v, t2v, t3v])):
                acum = round(round(t1v,1)*0.30 + round(t2v,1)*0.30 + round(t3v,1)*0.40, 1)
                if acum < 4.0:
                    n_bajo += 1
    materias_en_bajo.append(n_bajo)

# Armar resultado clasificación
df_result = df_predict[['student_id', 'grade',
                          'section_anon']].copy()
df_result['pred_label']      = pred_labels
df_result['proba_critical']  = proba_critical
df_result['proba_recovery']  = proba_recovery
df_result['proba_riesgo']    = proba_riesgo
df_result['confianza']       = confianza
df_result['n_bajo_acumulada']    = materias_en_bajo
df_result['t3_confirma_riesgo']  = (
    df_result['n_bajo_acumulada'] >= 3
)

def asignar_categoria(row):
    modelo_critico  = row['pred_label'] == 'critical'
    modelo_recovery = row['pred_label'] == 'recovery'
    t3_riesgo       = row['t3_confirma_riesgo']

    if modelo_critico and t3_riesgo:
        return 'Riesgo Confirmado'
    elif not modelo_critico and t3_riesgo:
        return 'Punto Ciego'
    elif modelo_critico and not t3_riesgo:
        return 'Riesgo Teórico'
    else:
        return 'Sin Riesgo'

df_result['categoria']       = df_result.apply(
    asignar_categoria, axis=1)
df_result['umbral_usado']    = UMBRAL_PRODUCCION
df_result['modelo']          = 'Random Forest v2'
df_result['fecha_ejecucion'] = datetime.now().strftime('%Y-%m-%d')

print(f"\n✓ Categorías asignadas:")
for cat in ['Riesgo Confirmado', 'Punto Ciego',
            'Riesgo Teórico', 'Sin Riesgo']:
    n = (df_result['categoria'] == cat).sum()
    print(f"  {cat}: {n} ({n/len(df_result)*100:.1f}%)")

print(f"\n── Rango proba_riesgo ──")
print(f"  Min: {proba_riesgo.min():.3f}")
print(f"  Max: {proba_riesgo.max():.3f}")
print(f"  Media: {proba_riesgo.mean():.3f}")

# Guardar
spark.createDataFrame(df_result).write\
    .mode("overwrite")\
    .parquet(f"{SILVER}/predictions_25_26_v2")
print(f"✓ Clasificación guardada")

In [0]:
# ── PASO 2: Regresión T3 + Intervalos ──
print("\n" + "=" * 60)
print("PASO 2 — Regresión T3 + Intervalos P10-P90")
print("=" * 60)

X_reg = pd.DataFrame(
    imp_reg.transform(df_predict[feats_reg].fillna(0)),
    columns=feats_reg
)

# Predicción puntual
Y_pred = rf_reg.predict(X_reg)
df_t3  = pd.DataFrame(
    Y_pred,
    columns=[c.replace('_T3','_T3_pred') for c in T3_COLS]
).round(2).clip(1.0, 7.0)

df_t3['student_id']   = df_predict['student_id'].values
df_t3['grade']        = df_predict['grade'].values
df_t3['section_anon'] = df_predict['section_anon'].values

# Confiabilidad por materia
for mat, info in confiabilidad.items():
    df_t3[f'{mat}_T3_pred_confiable'] = info['confiable']

# Intervalos P10-P90
print("  Calculando intervalos P10-P90...")
tree_preds = np.array([
    t.predict(X_reg) for t in rf_reg.estimators_
])
p10 = np.percentile(tree_preds, 10, axis=0).clip(1.0, 7.0)
p90 = np.percentile(tree_preds, 90, axis=0).clip(1.0, 7.0)
amp = p90 - p10

df_int = pd.DataFrame(
    p10,
    columns=[c.replace('_T3','_T3_p10') for c in T3_COLS]
)
for i, col in enumerate(T3_COLS):
    df_int[col.replace('_T3','_T3_p90')] = p90[:,i]
    df_int[col.replace('_T3','_T3_amp')] = amp[:,i]

df_int['student_id']            = df_predict['student_id'].values
df_int['grade']                 = df_predict['grade'].values
df_int['section_anon']          = df_predict['section_anon'].values
df_int['incertidumbre_promedio'] = amp.mean(axis=1).round(3)

spark.createDataFrame(df_t3).write\
    .mode("overwrite")\
    .parquet(f"{SILVER}/t3_predictions_25_26")
spark.createDataFrame(df_int).write\
    .mode("overwrite")\
    .parquet(f"{SILVER}/t3_intervals_25_26")

print(f"✓ Regresión T3 guardada")
print(f"✓ Intervalos P10-P90 guardados")

In [0]:
# ── PASO 3: Clustering dinámico ──
print("\n" + "=" * 60)
print("PASO 3 — Clustering dinámico (K-Means K=2)")
print("Recalculado sobre datos actuales")
print("=" * 60)

FEATURES_CLUSTER = [
    'avg_T1', 'avg_T2', 'tendencia_general',
    'n_bajo_T1', 'n_bajo_T2', 'dispersion_T2',
    'min_nota_T2', 'n_destacadas_T2',
    'total_absences', 'ratio_ausencia_clase',
    'n_f1', 'n_f2', 'indice_disciplinario',
]
available_cluster = [f for f in FEATURES_CLUSTER
                     if f in df_predict.columns]

X_cluster = df_predict[available_cluster].copy()
X_cluster = X_cluster.fillna(X_cluster.mean())

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# K-Means K=2
km       = KMeans(n_clusters=2, random_state=42,
                   n_init=10, max_iter=300)
clusters = km.fit_predict(X_scaled)

# Nombrar clusters por características
# (robusto a inversión de etiquetas entre runs)
avg_global = df_predict['avg_T2'].mean()
nombres_cluster = {}
for k in range(2):
    mask  = clusters == k
    avg_k = df_predict.loc[mask, 'avg_T2'].mean()
    if avg_k >= avg_global:
        nombres_cluster[k] = 'Rendimiento sólido'
    else:
        nombres_cluster[k] = 'Riesgo multidimensional'

df_cluster = df_predict[['student_id', 'grade',
                           'section_anon']].copy()
df_cluster['cluster'] = clusters
df_cluster['perfil']  = df_cluster['cluster'].map(
    nombres_cluster
)

# Variables descriptivas para el dashboard
vars_desc = [
    'avg_T1', 'avg_T2', 'tendencia_general',
    'n_bajo_T1', 'n_bajo_T2', 'total_absences',
    'n_f1', 'n_f2', 'indice_disciplinario',
    'n_destacadas_T2', 'min_nota_T2'
]
for v in vars_desc:
    if v in df_predict.columns:
        df_cluster[v] = df_predict[v].values

print(f"✓ Clustering completado")
for k in range(2):
    n = (clusters == k).sum()
    print(f"  {nombres_cluster[k]}: {n} estudiantes")

spark.createDataFrame(df_cluster).write\
    .mode("overwrite")\
    .parquet(f"{SILVER}/clusters_25_26")
print(f"✓ Clusters guardados")

print(f"\n{'='*60}")
print(f"03_apply_model COMPLETO ✓")
print(f"{'='*60}")
print(f"  Clasificación → predictions_25_26_v2")
print(f"  Regresión T3  → t3_predictions_25_26")
print(f"  Intervalos    → t3_intervals_25_26")
print(f"  Clusters      → clusters_25_26")
print(f"  Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M')}")